# Validation of the quantum firm-name harmonisation (LLM labels)

The quantum applicants have no gold standard, so we build a small one with LLMs and score the harmonisation with **the same pairwise metric as the course gold standard** (`precision_recall_f1`: a pair counts when both names are in the same group).

**Procedure**

| Step | What happens |
|---|---|
| 1 | Read `quantum_harmonised_PROD.csv`; group by harmonised name (`name_after`) and count the distinct raw names and the families of every cluster |
| 2 | Select **30 clusters**: the 10 with most raw names (the most harmonised), the 10 with most families, 10 at random (only clusters with at least 2 raw names, which are the only ones that contain pairs) |
| 3 | Print the 30 harmonised names; take **all** their rows from the harmonised file |
| 4 | Write two files: the **key** (raw name, cleaned name, harmonised name, cluster number) and the **file for the LLMs** (only `id` and the raw name, rows shuffled, empty `truth` column) |
| 5 | Each LLM fills `truth`: the same number for names of the same organisation |
| 6 | `precision_recall_f1(truth, cluster)` as in the gold standard, plus precision per cluster and agreement between LLMs |

**Two choices to keep the test honest.** The LLM never sees `name_after`, the cleaned name or the cluster number (otherwise it anchors to our groups), and the rows are shuffled so that cluster members are not adjacent.

**What the test can and cannot say.** The sample contains only names that *we* put in the same cluster, so it measures **precision** (are the merges right?). Names that should have been merged but sit in other clusters are not in the sample, hence **recall is close to 1 by construction and F1 is overstated**: read precision, not F1.

In [16]:
from math import comb
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Config ----
HARMONISED_FILE = "quantum_harmonised_PROD.csv"   # output of the harmonisation notebook
N_TOP_NAMES, N_TOP_FAMILIES, N_RANDOM = 3, 3, 3
MIN_NAMES = 2          # only clusters with >= 2 distinct raw names (single-name clusters contain no pairs)
SEED = 42              # fixed seed: same sample at every run

KEY_FILE = "validation_sample_key.csv"            # with cluster and harmonised name: NOT for the LLMs
LLM_INPUT_FILE = "validation_for_llm.csv"         # id + raw name only: for the LLMs
LLM_FILES = ["validation_llm_1.csv"]              # filled files, one per LLM, columns: id, truth

## Step 1 - Choose the 9 clusters

The families of a cluster are the sum of the `families` of its rows. The harmonised file has one row per raw name and no family ids, so a family that appears under two names of the same cluster is counted twice: the number is used only to rank clusters.

In [18]:
q = pd.read_csv(HARMONISED_FILE)
g = (q.groupby("name_after").agg(n_names=("name_before", "nunique"), families=("families", "sum")).reset_index())
cand = g[g["n_names"] >= MIN_NAMES]                                      # clusters that contain pairs
print(f"{len(q):,} raw names -> {len(g):,} harmonised names | with >= {MIN_NAMES} raw names: {len(cand):,}")

top_names = cand.sort_values(["n_names", "families"], ascending=False).head(N_TOP_NAMES).assign(selection="most names")
rest = cand[~cand["name_after"].isin(top_names["name_after"])]           # never pick the same cluster twice
top_fam = rest.sort_values(["families", "n_names"], ascending=False).head(N_TOP_FAMILIES).assign(selection="most families")
rest = rest[~rest["name_after"].isin(top_fam["name_after"])]
rand = rest.sample(min(N_RANDOM, len(rest)), random_state=SEED).assign(selection="random")

chosen = pd.concat([top_names, top_fam, rand], ignore_index=True)
chosen["cluster"] = np.arange(1, len(chosen) + 1)                        # cluster number
print(f"\nSelected clusters: {len(chosen)}")
print(chosen[["cluster", "selection", "name_after", "n_names", "families"]].to_string(index=False))

2,184 raw names -> 1,073 harmonised names | with >= 2 raw names: 362

Selected clusters: 9
 cluster     selection                          name_after  n_names  families
       1    most names           SILICON QUANTUM COMPUTING       93       400
       2    most names                        ZHEJIANG LAB       63        93
       3    most names ORIGIN QUANTUM COMPUTING TECH HEFEI       39      1055
       4 most families IBM INTERNATIONAL BUSINESS MACHINES       16      1206
       5 most families   BEIJING BAIDU NETCOM SCIENCE TECH        4       413
       6 most families            MICROSOFT TECH LICENSING        6       287
       7        random                          QET SWEDEN        2         6
       8        random                      BOE TECH GROUP        3         5
       9        random                             RED HAT        4       139


## Step 2 - Build the key file and the file for the LLMs

All the rows of the 30 clusters are taken from the harmonised file. The **key** keeps the raw name, the cleaned name, our harmonised name and the cluster number. The **file for the LLMs** keeps only a random `id` and the raw name, in shuffled order, with an empty `truth` column.

In [19]:
sample = q[q["name_after"].isin(chosen["name_after"])].merge(chosen[["name_after", "cluster", "selection"]], on="name_after")
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle: members of a cluster are not adjacent
sample["id"] = np.arange(1, len(sample) + 1)

key = sample[["id", "name_before", "name_clean", "name_after", "cluster", "selection", "families"]]
key.to_csv(KEY_FILE, index=False)                                         # our side: keep it away from the LLMs

for_llm = sample[["id", "name_before"]].assign(truth="")                  # nothing that reveals our clusters
for_llm.to_csv(LLM_INPUT_FILE, index=False)
print(f"{len(sample)} names in {sample['cluster'].nunique()} clusters | written {KEY_FILE} (key) and {LLM_INPUT_FILE} (for the LLMs)")

230 names in 9 clusters | written validation_sample_key.csv (key) and validation_for_llm.csv (for the LLMs)


## Step 3 - Ask the LLMs

Give each LLM `validation_for_llm.csv` together with the instruction below, and save each answer as `validation_llm_1.csv`, `validation_llm_2.csv`, ... (columns `id`, `truth`; list the files in `LLM_FILES`).

> The attached CSV lists 1 name per row (`id`, `name_before`) of patent applicants in quantum computing. Fill the `truth` column with an integer: **give the same number to names that refer to the same organisation and different numbers to different organisations**. Return the same CSV with `id` and `truth`, keeping all rows.
>
> Rules (the same as the course gold standard):
> - **Same group**: the same company written in different ways (legal form, case, punctuation, typos, transliteration, translation, **a name in Chinese/Japanese and its Latin version**) and subsidiaries or branches that carry the parent's name (e.g. `IBM Deutschland` and `IBM`, `Nokia USA` and `Nokia Technologies`).
> - **Different groups**: companies that only share a generic word (`Quantum`, `Technology`, `Computing`, a city or province such as `Hefei`, `Shenzhen`) or a prefix, and separate companies of the same conglomerate under another brand (e.g. `Mitsubishi Electric` and `Mitsubishi Chemical`).
> - Use the names only; if you are unsure whether two names are the same organisation, give them different numbers.
> - Group numbers are free: they only need to be consistent within the file.

## Step 4 - Metrics

`precision_recall_f1(gold, pred)` is the function of the gold-standard notebook, with the LLM's `truth` as gold and our `cluster` as prediction.

* **Pairwise precision** (main number): of the pairs of names we put together, how many the LLM also puts together.
* **Precision per cluster**: a large cluster has many pairs (`n(n-1)/2`) and dominates the pooled number, so each cluster is also shown alone. The macro average gives every cluster the same weight.
* **Agreement between LLMs** (with 2 or more files): F1 between the two `truth` columns. If it is low the labels are unreliable, and the disagreeing clusters should be checked by hand.
* Recall and F1 are printed for consistency with the course metric, but see the note at the top: recall is not informative on this sample.

In [24]:
from math import comb

def pairs_inside(sizes):
    return sum(comb(int(n), 2) for n in sizes)

def precision_recall_f1(gold_groups, predicted_groups):
    "Same function as the gold-standard notebook: a pair counts if both names are in the same group."
    t = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(t.groupby(["gold", "pred"]).size())        # pairs together in both truth and prediction
    tp_fp = pairs_inside(t.groupby("pred").size())               # pairs together in the prediction
    tp_fn = pairs_inside(t.groupby("gold").size())               # pairs together in the truth
    p, r = (tp / tp_fp if tp_fp else 1.0), (tp / tp_fn if tp_fn else 0.0)
    return {"groups found": t["pred"].nunique(), "precision": p, "recall": r, "F1": 2 * p * r / (p + r) if p + r else 0.0}

key = pd.read_csv("validation_sample_key.csv")                      # id, name_before, name_clean, name_after, cluster, selection, families
llm = pd.read_csv("validation_harmonised.csv")[["id", "truth"]] # id, name, truth (filled by the LLM)

m = key.merge(llm, on="id", how="left")
assert m["truth"].notna().all(), "every id needs a truth value"
print(precision_recall_f1(m["truth"], m["cluster"]))            # truth = gold, cluster = our prediction

{'groups found': 9, 'precision': 0.23459216622209744, 'recall': 0.9976119402985074, 'F1': 0.37985905887701754}
